# Data exploration — Egyptian Hieroglyph dataset

Goal: understand the raw dataset in `data/raw/` before writing any
training code. We want to know: how many classes, how balanced are they,
what do the images actually look like, and what train/val/test split
strategy makes sense given what we find.

In [ ]:
from pathlib import Path
from collections import Counter
import random

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

DATA_ROOT = Path("..") / "data" / "raw"
assert DATA_ROOT.exists(), f"Expected dataset at {DATA_ROOT.resolve()}"

## Class distribution

Each subfolder of `data/raw/` is a Gardiner sign class. Count images per class.

In [ ]:
class_dirs = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir()])
counts = Counter()
for cls_dir in class_dirs:
    n = sum(1 for f in cls_dir.iterdir() if f.is_file())
    counts[cls_dir.name] = n

df = pd.DataFrame({"class": list(counts.keys()), "count": list(counts.values())})
df = df.sort_values("count", ascending=False).reset_index(drop=True)

print(f"Classes: {len(df)}")
print(f"Total images: {df['count'].sum()}")
print(df["count"].describe())

In [ ]:
# Classes with very few images are a real risk for stratified train/val/test
# splitting later (need at least 1 sample per split, ideally more).
print("Classes with fewer than 5 images:")
print(df[df["count"] < 5])

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(df)), df["count"])
ax.set_xlabel("Class (sorted by count, descending)")
ax.set_ylabel("Number of images")
ax.set_title("Class distribution — Gardiner sign classes")
plt.tight_layout()
plt.show()

## Sample images

Look at actual glyphs, not just counts — sanity check that images look like
what we expect (small grayscale glyph crops).

In [ ]:
random.seed(0)
sample_classes = random.sample(class_dirs, 12)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, cls_dir in zip(axes.flat, sample_classes):
    img_path = next(f for f in cls_dir.iterdir() if f.is_file())
    img = Image.open(img_path)
    ax.imshow(img, cmap="gray")
    ax.set_title(cls_dir.name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Image size / format consistency

Check whether all images share the documented spec (75x50 grayscale), or if
there's variation we need to handle in the transforms.

In [ ]:
sizes = Counter()
modes = Counter()
for cls_dir in class_dirs:
    for f in cls_dir.iterdir():
        if not f.is_file():
            continue
        with Image.open(f) as img:
            sizes[img.size] += 1
            modes[img.mode] += 1

print("Sizes:", sizes.most_common(5))
print("Modes:", modes.most_common(5))

## Decisions for the training pipeline

- **Input size:** resize to 224x224 to match ResNet18's expected input
  (pretrained on ImageNet at that resolution).
- **Normalization:** ImageNet mean/std, since we're fine-tuning an
  ImageNet-pretrained backbone — grayscale images will be replicated to 3
  channels first.
- **Split strategy:** stratified train/val/test, so each class is
  represented proportionally in each split. The rare-class check above
  determines whether a plain stratified split is safe or whether we need to
  fold ultra-rare classes into train-only / exclude them — decide after
  seeing the actual counts.

In [ ]:
from sklearn.model_selection import train_test_split

# Build a flat (path, label) list.
samples = []
for cls_dir in class_dirs:
    for f in cls_dir.iterdir():
        if f.is_file():
            samples.append((str(f), cls_dir.name))

paths, labels = zip(*samples)

# 70/15/15 stratified split. If this raises (some class too small to
# stratify), that confirms we need to handle rare classes specially.
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    paths, labels, test_size=0.3, stratify=labels, random_state=0
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=0
)

print(f"train: {len(train_paths)}  val: {len(val_paths)}  test: {len(test_paths)}")